# 02 · Evaluación, imputación y análisis estadístico
**¿Qué método reconstruye una concentración conocida y qué cambia al completar las que faltan?** Las primeras secciones usan mediciones de Ishii ocultadas. Solo después se consultan los huecos reales. Si ya existe una ejecución íntegra, el cuaderno lee sus resultados. En la primera ejecución los genera en su equipo; puede tardar varios minutos. Para regenerarlo todo: `python -m protocolo.revision --procesos 4`.

La preparación y las unidades se explican en el [cuaderno 01](01_auditoria_preparacion.ipynb). `METABOLITO` permite cambiar la selección de las tablas y el gráfico individual.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from protocolo.datos import PRIV, PUB, leer_matriz, preparar
from protocolo.distribuciones import matrices
from protocolo.estrategias import PRINCIPALES
from protocolo.figuras import por_metabolito

from IPython.display import Image, display
from protocolo.figuras import FIG

def mostrar_figura(nombre):
    display(Image(filename=str(FIG / (nombre + ".png"))))

METABOLITO = "fum"
from protocolo.estrategias import RES
from protocolo.revision import resultados_vigentes

# Un proceso independiente permite ejecutar los modelos también desde Jupyter en Windows.
if not resultados_vigentes():
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "protocolo.revision", "--procesos", "4"], check=True)

def tabla(nombre):
    carpeta = PUB if nombre == "MMC2_pareado.csv" else RES
    return pd.read_csv(carpeta / nombre)

resultados = tabla("comparacion_estrategias.csv")
huecos = tabla("estimaciones_por_casilla.csv.gz")
if not (PRIV / "ishii.csv").exists():
    preparar()
ishii = leer_matriz("ishii")

## 1. Probar antes de completar
Se apartan grupos de perturbación completos (cinco bloques) o una serie analítica completa y sus grupos relacionados. Se oculta una concentración medida, o varias que suelen faltar juntas en el entrenamiento. La máscara completa de 135 columnas no se pudo reproducir; la prueba conjunta usa las cuatro anclas. Las cuatro estimaciones fijas usan Ishii y los mismos predictores candidatos; las opciones con MMC2 y covariables conservan sus controles pareados. Nunca se usa Gerosa como verdad. Los tres métodos son `SimpleImputer`, `KNNImputer` (3 o 5 vecinos) e `IterativeImputer` con BayesianRidge.

La tabla separa el error en **casillas comunes** de la cobertura sobre **todos los intentos**. Micro da el mismo peso a cada casilla y macro da el mismo peso a cada metabolito.

In [ ]:
def comparar(prueba, mascara):
    t = resultados[(resultados.prueba == prueba) & (resultados.tipo == mascara)]
    comunes = t[t.alcance == "comunes"].set_index("estrategia")
    disponibles = t[t.alcance == "disponibles"].set_index("estrategia")
    columnas = ["predichas", "MAE_micro_mM", "MAE_macro_mM", "RMSE_mM", "sesgo_mM"]
    return comunes[columnas].join(disponibles.cobertura.rename("cobertura_total")).reindex(
        PRINCIPALES + ["Ideal_no_aplicable"])
display(comparar("perturbacion", "individual").round(4))
display(comparar("serie", "individual")[["predichas", "MAE_micro_mM", "cobertura_total"]].round(4))
# Para el ocultamiento simultáneo: comparar("perturbacion", "conjunta_anclas").

In [ ]:
# Distribución del error absoluto, residuos y comparación con la medición
mostrar_figura("06_errores_externos")

In [ ]:
# Errores frente a concentración, dispersión, faltantes, predictores, donantes y grupo experimental
mostrar_figura("07_residuos_y_soporte")

## 2. ¿Ayuda elegir un método por casilla?
La regla original exige diez cultivos, cinco grupos y dos bloques internos y que la ventaja se repita ante cada competidor. Cuando falta evidencia, usa con frecuencia la mediana. Se ensayan dos respaldos (KNN3 o un método elegido internamente) y una jerarquía: perfil → metabolito → metabolitos estadísticamente parecidos → conjunto del entrenamiento. La complejidad y el umbral se deciden mediante pruebas dentro del entrenamiento. El límite ideal elige *después de conocer la verdad*: nunca se aplica a un hueco.

El «aporte al MAE» divide la diferencia acumulada de un subconjunto por **todas** las casillas comunes. Por tanto, los aportes de estados distintos se pueden sumar.

In [ ]:
diag = tabla("descomposicion_regla.csv")
display(diag.query("prueba == 'perturbacion/individual' and dimension == 'estado'")[
    ["valor", "n", "MAE_regla_mM", "MAE_KNN3_mM", "aporte_al_MAE_mM"]].round(4))
pares = tabla("diferencias_pareadas.csv")
display(pares.query("tipo == 'individual' and a == 'KNN3' and b in ['Original', 'KNN5', 'Jerarquica']")[
    ["prueba", "a", "b", "n", "delta_MAE_mM", "IC95_inf", "IC95_sup"]].round(4))

In [ ]:
# Elección, respaldos, métodos alternativos en las mismas casillas y límite ideal
mostrar_figura("08_eleccion_y_respaldo")

Una diferencia positiva en la segunda tabla indica mayor error de A que de B. Cada réplica del intervalo conserva juntas las mediciones de una perturbación. No se observa una ventaja constante de la jerarquía frente a los métodos fijos. El aporte de MMC2 también depende de la sustancia; se compara únicamente con el control que emplea el mismo panel.

In [ ]:
externos = tabla("MMC2_pareado.csv")
display(externos[externos.objetivo.isin([METABOLITO, "pep"])].head(8).round(4))
riesgo = tabla("seleccion_interna.csv")
display(riesgo[riesgo.particion.eq("F")][["configuracion", "MAE_interno_mM", "casillas", "grupos", "elegida"]].round(4))

## 3. ¿Qué representa un intervalo del modelo?
Cada casilla iterativa utiliza 200 sorteos independientes (100 con proteínas). Los percentiles del 2,5 % y el 97,5 % forman el intervalo nominal del 95 %. La SD de esos sorteos describe **una predicción**. La SD experimental describe **diferencias entre cultivos**; la SD de los residuos describe **errores**; el intervalo de bootstrap describe **la precisión de una diferencia de MAE**. Solo las casillas ocultas permiten medir calibración.

In [ ]:
c = tabla("calibracion_posterior.csv")
display(c.query("tipo == 'individual' and nominal == 0.95")[["prueba", "n", "grupos", "dentro", "observada", "ancho_mediano_mM"]].round(4))
fijo = tabla("calibracion_iterativo_fijo.csv")
display(fijo[fijo.objetivo.eq("todos")].round(4))

In [ ]:
# Sorteos posteriores, intervalos, calibración y asociación de amplitud con el error
mostrar_figura("09_posterior_y_calibracion")

## 4. Huecos reales: qué se estima y qué evidencia lo respalda
Se mantienen las 3.402 mediciones. Hay 648 huecos y los blancos no equivalen a cero. Se separan método fijo, elección con evidencia local, respaldo general y pendiente. La cobertura por sí sola no indica exactitud. «Comparable» significa que para la misma estrategia, opción, método, objetivo y perfil hubo al menos diez cultivos, cinco grupos y dos bloques con una verdad oculta. El error de otros cultivos no revela el error de una casilla auténticamente vacía.

In [ ]:
evidencia = tabla("evidencia_por_metabolito.csv")
display(evidencia[evidencia.objetivo.isin([METABOLITO, "cit", "pep", "s7p"])].round(4))
vista = huecos[huecos.objetivo.eq(METABOLITO) & huecos.estrategia.eq("KNN5")]
display(vista[["cultivo", "source_cell", "pred", "estado", "motivo", "evidencia_en_prueba", "n_evaluadas"]].round(4))
# 'pred' es una estimación en mM. Una tabla vacía indica que no hay huecos para ese metabolito.

## 5. Comparar lo medido con la tabla completada
El histograma y la ECDF distinguen las distribuciones originales y completadas. Los rombos señalan solo las estimaciones. El citrato tiene 18 huecos; el ATP, ninguno. Los demás métodos pueden consultarse con el argumento `estrategias`.

In [ ]:
completadas = dict(matrices(ishii, huecos))
por_metabolito(METABOLITO, ish=ishii, completadas=completadas)
plt.show()

In [ ]:
impacto = tabla("efecto_distribucion.csv")
display(impacto.query("alcance == 'huecos_comunes'")[["estrategia", "imputados", "metabolitos_con_imputacion", "mediana_cociente_SD", "en_mediana"]].round(3))
detalle = tabla("estadistica_por_estrategia.csv.gz")
display(detalle[(detalle.objetivo == METABOLITO) & (detalle.alcance == "disponibles")][
    ["estrategia", "observados", "imputados", "faltantes", "media", "mediana", "SD", "cociente_SD", "IQR"]].round(4))

In [ ]:
# Media, mediana, SD, CV e IQR con líneas de identidad y cambio de SD
mostrar_figura("03_estadisticos_antes_despues")

## 6. ¿Se conservan las relaciones?
Una correlación puede cambiar porque hay más pares disponibles al completar los huecos. Sobre las mismas parejas observadas la correlación original se conserva, pues no se altera ningún valor medido. Para medir recuperación se reconstruyen por separado las mediciones conocidas y se comparan exactamente las mismas casillas y pares con los originales. No es una simulación de borrar toda la matriz a la vez. El PCA solo describe cuatro anclas; se ajusta en los cultivos originalmente completos y se proyectan los nuevos sin volver a ajustar los ejes.

In [ ]:
# Correlaciones originales, completadas y sus diferencias para Pearson y Spearman
mostrar_figura("04_correlaciones")

In [ ]:
# Correlaciones bivariadas y proyección sobre los mismos componentes
mostrar_figura("05_pares_y_pca")

In [ ]:
recuperacion = tabla("recuperacion_experimental.csv")
display(recuperacion[["estrategia", "celdas", "mediana_cociente_SD", "pares_spearman", "mediana_abs_delta_spearman"]].round(3))

**Interpretación.** KNN es una referencia útil cuando hay al menos dos predictores compartidos y tres donantes con el objetivo medido. El iterativo conserva mejor la dispersión en el ocultamiento, aunque no siempre las correlaciones ni la cobertura nominal del intervalo. La mediana aporta solo el centro del entrenamiento cuando se usa como respaldo. Si se necesita evidencia específica del cultivo y no existe, conviene dejar la casilla pendiente. Los blancos potencialmente asociados al límite de detección pueden comportarse de otra forma. Para balances, proteínas y flujos, conviene repetir el análisis con solo mediciones y con cada matriz de sensibilidad antes de interpretar una relación como biológica.